# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [ ]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [ ]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

Define function for loading data

In [ ]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [ ]:
#export
def clean_data(df):
    """
    Cleans the data by casting columns to the correct data types.
    """
    return df.withColumn("passenger_count", col("passenger_count").cast("integer")) \
             .withColumn("total_amount", col("total_amount").cast("float")) \
             .withColumn("tip_amount", col("tip_amount").cast("float")) \
             .withColumn("trip_distance", col("trip_distance").cast("float")) \
             .withColumn("fare_amount", col("fare_amount").cast("float")) \
             .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime"))) \
             .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))


### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [ ]:
#export
def common_pair(df):
    """
    Finds the top 10 pickup-dropoff location pairs with the highest total passenger count.
    Also calculates the per-person rate for each pair.
    """
    df_filtered = df.filter(col("PULocationID") != col("DOLocationID"))

    df_grouped = df_filtered.groupBy("PULocationID", "DOLocationID") \
                            .agg({"passenger_count": "sum", "total_amount": "sum"}) \
                            .withColumnRenamed("sum(passenger_count)", "total_passenger_count") \
                            .withColumnRenamed("sum(total_amount)", "total_fare")

    df_final = df_grouped.withColumn("per_person_rate", col("total_fare") / col("total_passenger_count")) \
                         .orderBy(col("total_passenger_count").desc(), col("per_person_rate").desc()) \
                         .select("PULocationID", "DOLocationID", "total_passenger_count", "per_person_rate")

    return df_final.limit(10)


### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [ ]:
#export
def distance_with_most_tip(df):
    """
    Finds the top 15 trip distances where the tip percentage is the highest.
    """
    df_filtered = df.filter((col("fare_amount") > 2) & (col("trip_distance") > 0))

    df_tips = df_filtered.withColumn("tip_percent", (col("tip_amount") * 100) / col("fare_amount"))

    df_grouped = df_tips.withColumn("trip_distance", ceil(col("trip_distance"))) \
                        .groupBy("trip_distance") \
                        .agg({"tip_percent": "avg"}) \
                        .withColumnRenamed("avg(tip_percent)", "tip_percent") \
                        .orderBy(col("tip_percent").desc())

    return df_grouped.limit(15)


### Q1.4

Determine the average speed at different times of day.

In [ ]:
#export
def time_with_most_traffic(df):
    """
    Determines which hour of the day has the most traffic based on average speed.
    """
    df_filtered = df.withColumn("trip_duration", 
                                (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 3600)

    df_filtered = df_filtered.filter(col("trip_duration") > 0)

    df_grouped = df_filtered.groupBy(hour("tpep_pickup_datetime").alias("time_of_day")) \
                            .agg({"trip_distance": "avg", "trip_duration": "avg"}) \
                            .withColumnRenamed("avg(trip_distance)", "avg_distance") \
                            .withColumnRenamed("avg(trip_duration)", "avg_duration")

    df_final = df_grouped.withColumn("avg_speed", col("avg_distance") / col("avg_duration")) \
                         .select("time_of_day", "avg_speed") \
                         .orderBy(col("time_of_day"))

    return df_final


## The below cells are for you to investigate your solutions and will not be graded

In [ ]:
df = load_data()
df = clean_data(df)

In [ ]:
common_pair(df).show()

In [ ]:
distance_with_most_tip(df).show()

In [ ]:
time_with_most_traffic(df).show()